# formal_30_v4 剩余两个复杂样本：开发回归与事后人工修复结案

记录日期：2026-10-27。对象为已参与调优的 `TaskManager` 与 `MonitoringDashboard`，均属于重复开发集回归，不是未见过的独立测试集。原批次结果、事后候选和人工确认必须分开统计。


## 1. 输入与自动批次结果

配置：`migration_pipeline/batch_configs/formal_30_v4_generalization_remaining_v1.json`。原始 checkpoint：`data/experiments/batch_reports/formal_30_v4_generalization_remaining_v1_summary.json`。两个输入 Vue 文件来自冻结的 `formal_30_v3` 协议。

自动批次完成 2/2，**通过 0/2、失败 2/2**，每个样本均达到 3 轮修复上限。生成 203,764 tokens、修复 507,480 tokens、合计 711,244 tokens。该 checkpoint 未被事后修复覆盖。

| 样本 | 自动最终结果 | 最终错误 | tokens | 冻结失败证据 |
| --- | --- | --- | ---: | --- |
| TaskManager | 失败 | `invalid_san_event_handler_syntax`、`invalid_san_computed_instance_method_call` | 337,853 | `data/experiments/snapshots/formal_30_v4_generalization_remaining_v1_task_manager_failed_20261027_195349/manifest.json` |
| MonitoringDashboard | 失败 | `invalid_san_event_handler_syntax`、`invalid_san_for_alias_syntax`、`duplicate_class_attribute`、`san_script_compile_error`；San 渲染失败 | 373,391 | `data/experiments/snapshots/formal_30_v4_generalization_remaining_v1_monitoring_failed_20261027_205900/manifest.json` |

TaskManager 的结构相似度 0.9889 和视觉检查通过不能覆盖静态语义错误。MonitoringDashboard 在脚本编译阶段失败，因此其 0.0 结构相似度不是可解释的视觉保真度分数。


## 2. TaskManager 的事后修复链

冻结失败后先制作有边界的 v1 候选：修正 San 事件语法和 computed 对实例方法的调用。静态校验、编译、轻量视觉通过，但人工浏览器检查失败，故没有将其记作通过。v2 候选继续处理 watcher、子组件事件载荷、拖拽索引、弹窗点击隔离、表单初始值以及设置/标签状态。v2 的针对性测试、静态校验、编译和轻量视觉通过；用户随后确认人工通过。

v2 San SHA-256：`924c33e224776152e9533469e68c32f897a08581a62fee0cc5995ff89e0cbb22`。候选及确认快照：`data/experiments/snapshots/formal_30_v4_remaining_task_manager_runtime_v2_manual_pass_20261027_203800/manifest.json`。详细诊断：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261027/task_manager_remaining_v1_review.json`。


## 3. MonitoringDashboard 的事后修复链

冻结的自动产物包含七个以连字符命名的非法 JavaScript 内联组件变量，以及 Vue 风格事件绑定、tuple 循环别名和重复 `class`，导致脚本无法编译。源哈希约束的 v1 重放修复上述语法/结构问题，并显式传递图表依赖和子组件事件载荷；静态校验、San 编译及轻量视觉通过（结构相似度 0.9912）。但代码复核发现 SettingsForm 没有从传入的 `settings` prop 初始化，故 v1 未交付人工验收。

v2 另行恢复 SettingsForm 初始值语义；针对性测试 4/4、全套单测 149/149 通过，静态校验、San 编译及轻量视觉通过（0.9912）。用户在 Vue/San 双测试页上确认 v2 人工通过。v2 San SHA-256：`6847de423b95ef768f8c7705ba935445bc8d26a03fa6bddf1f5cc8e1b656d7c1`。确认快照：`data/experiments/snapshots/formal_30_v4_remaining_monitoring_settings_v2_manual_pass_20261027_215100/manifest.json`；诊断：`data/experiments/analysis/formal_30_v4_generalization_gap_audit_20261027/monitoring_remaining_v1_review.json`。


## 4. 可报告结论与限制

- 本开发回归批次的自动通过率仍为 **0/2**；不能把两个事后候选的人工通过回填到自动 checkpoint。
- 两个事后候选均获用户人工确认，但没有逐场景日志或截图，因此只记录 `passed_by_user`，不扩写为逐项可追溯通过。
- 这些结果证明具体修复候选可供人工使用，不证明通用生成/修复规则已在新批次自动产出同等代码，更不是独立测试集效果。
- 下一阶段应把有边界的通用语法/结构规则提炼到生成、校验或修复阶段，加入跨样本正反例，再做新的开发集自动回归。交互语义仍需单独场景验证；不能将业务字段硬编码成通用规则。仅在开发阶段收敛并冻结后，才运行未参与调优的独立测试集。
